# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
# Setup — same pattern as w03: getpass, never pasted, REL/TABLES as before
from getpass import getpass
import duckdb, os
import pandas as pd


HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
tbl = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

raw = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {tbl}),
    decision AS (SELECT max_d - INTERVAL 30 DAY AS decision_date FROM bounds),
    prior AS (
        SELECT f.content_hash_id,
            AVG(f.gsc_clicks)                                                 AS avg_daily_clicks_prior,
            AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0)     AS avg_position_prior,
            COUNT(*) FILTER (WHERE f.gsc_clicks > 0)                          AS days_with_clicks_prior,
            AVG(CASE WHEN f.ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_coverage_prior,
            SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.sessions_organic + f.sessions_ai), 0) AS ai_share_prior,
            COUNT(*) AS n_days_prior
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date - INTERVAL 90 DAY
          AND f.report_date <  d.decision_date
        GROUP BY f.content_hash_id
        HAVING COUNT(*) >= 30
    ),
    future AS (
        SELECT f.content_hash_id, AVG(f.gsc_clicks) AS avg_daily_clicks_future
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date AND f.report_date < d.decision_date + INTERVAL 30 DAY
        GROUP BY f.content_hash_id
    )
    SELECT p.*, fu.avg_daily_clicks_future
    FROM prior p JOIN future fu USING (content_hash_id)
""").df()

print(f'{len(raw):,} content items with full prior+future coverage')

# --- Missing-value handling (deliberate, not fillna(0) everywhere) ---
raw['no_ranking_data_prior'] = raw['avg_position_prior'].isna().astype(int)
raw['avg_position_prior'] = raw['avg_position_prior'].fillna(100)   # sentinel: worse than any real rank, never "best"
raw['ai_share_prior'] = raw['ai_share_prior'].fillna(0)             # zero organic+AI sessions -> treat AI share as 0

# --- Engineered features ---
raw['click_consistency_prior'] = raw['days_with_clicks_prior'] / raw['n_days_prior']  # normalizes window-length noise

# --- Categorical handling: position bucket, one-hot encoded ---
raw['position_bucket'] = pd.cut(
    raw['avg_position_prior'], bins=[-1, 10, 20, 50, 100],
    labels=['top10', 'p11_20', 'p21_50', 'beyond_50_or_unranked']
)
bucket_dummies = pd.get_dummies(raw['position_bucket'], prefix='pos')

feature_cols = ['avg_daily_clicks_prior', 'avg_position_prior', 'click_consistency_prior',
                 'ga4_coverage_prior', 'ai_share_prior', 'no_ranking_data_prior']
X = pd.concat([raw[feature_cols], bucket_dummies], axis=1)

raw['is_declining_future'] = (
    (raw['avg_daily_clicks_future'] < 0.75 * raw['avg_daily_clicks_prior']) &
    (raw['avg_daily_clicks_prior'] >= 1.0)
).astype(int)
y = raw['is_declining_future']

print('base rate:', y.mean())
X.head()

Paste your HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

363,626 content items with full prior+future coverage
base rate: 0.012686111554179294


,avg_daily_clicks_prior,avg_position_prior,click_consistency_prior,ga4_coverage_prior,ai_share_prior,no_ranking_data_prior,pos_top10,pos_p11_20,pos_p21_50,pos_beyond_50_or_unranked
0,0.0,100.000000,0.0,0.0,0.0,1,False,False,False,True
1,0.0,14.666667,0.0,0.0,0.0,0,False,True,False,False
2,0.0,100.000000,0.0,0.0,0.0,1,False,False,False,True
3,0.0,5.000000,0.0,0.0,0.0,0,True,False,False,False
4,0.0,100.000000,0.0,0.0,0.0,1,False,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
import pandas as pd
notes = pd.DataFrame([
    {'feature': 'avg_daily_clicks_prior', 'meaning': 'mean daily GSC clicks, prior 90d', 'missing': 'none (count-filtered)', 'available_before_decision': True},
    {'feature': 'avg_position_prior', 'meaning': 'mean search position, prior 90d, ranked days only', 'missing': f"filled 100 (sentinel) + no_ranking_data_prior flag, {raw['no_ranking_data_prior'].sum():,} rows", 'available_before_decision': True},
    {'feature': 'click_consistency_prior', 'meaning': 'share of prior-window days with any clicks', 'missing': 'none', 'available_before_decision': True},
    {'feature': 'ga4_coverage_prior', 'meaning': 'share of prior-window days GA4 data is trustworthy', 'missing': 'none', 'available_before_decision': True},
    {'feature': 'ai_share_prior', 'meaning': 'share of organic+AI sessions from AI surfaces', 'missing': f"filled 0, {raw['ai_share_prior'].isna().sum():,} rows had zero denominator", 'available_before_decision': True},
    {'feature': 'position_bucket (one-hot)', 'meaning': 'categorical rank tier from avg_position_prior', 'missing': 'n/a (derived post-fill)', 'available_before_decision': True},
])
notes

,feature,meaning,missing,available_before_decision
0,avg_daily_clicks_prior,"mean daily GSC clicks, prior 90d",none (count-filtered),True
1,avg_position_prior,"mean search position, prior 90d, ranked days only",filled 100 (sentinel) + no_ranking_data_prior ...,True
2,click_consistency_prior,share of prior-window days with any clicks,none,True
3,ga4_coverage_prior,share of prior-window days GA4 data is trustwo...,none,True
4,ai_share_prior,share of organic+AI sessions from AI surfaces,"filled 0, 0 rows had zero denominator",True
5,position_bucket (one-hot),categorical rank tier from avg_position_prior,n/a (derived post-fill),True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
print('Honest AUC:', honest_auc)

# Attack: smuggle the future-window value in directly
Xl = X.copy()
Xl['avg_daily_clicks_future'] = raw['avg_daily_clicks_future']
Xtrl, Xtel, _, _ = train_test_split(Xl, y, test_size=0.3, random_state=0, stratify=y)
leaky_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtrl, ytr).predict_proba(Xtel)[:, 1])
print('Leaky AUC (should jump toward 1.0):', leaky_auc)

# Correlation check: any real feature suspiciously tied to the label?
print(X.assign(label=y).corr()['label'].sort_values(ascending=False))

Honest AUC: 0.9949882309639908
Leaky AUC (should jump toward 1.0): 0.9995701731120061
label                        1.000000
click_consistency_prior      0.602058
avg_daily_clicks_prior       0.425197
ga4_coverage_prior           0.368675
pos_top10                    0.121204
pos_p11_20                   0.012039
ai_share_prior              -0.005707
pos_p21_50                  -0.025212
no_ranking_data_prior       -0.087915
pos_beyond_50_or_unranked   -0.096998
avg_position_prior          -0.111627
Name: label, dtype: float64


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- report_date, client_hash_id, content_hash_id — join/group keys, never model inputs
- scroll_events — present in ~0.76% of rows (per w03's density check); imputing would encode "did a ping fire," not real engagement
- avg_daily_clicks_future — label-derived by construction; Section 3 proves including it leaks

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.